In [1]:
import re

import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize
from transformers import AutoConfig, AutoModel, AutoTokenizer, pipeline
from tqdm.auto import tqdm

import src

In [2]:
pd.set_option("display.max_colwidth", 512)

In [3]:
nltk.download("vader_lexicon", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

True

In [4]:
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

pipe = pipeline("text-classification", model_name)
model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [ ]:
analyzer = SentimentIntensityAnalyzer()


STOPWORDS = set(stopwords.words("english"))


def tokenize(text: str) -> list[str]:
    return word_tokenize(text, language="english")


def clean(text: str, exclude_too: str):
    # unify text
    text = text.strip().lower().replace("\n", " ")

    # remove urls
    text = re.sub(r"(?:\@|http?\://|https?\://|www)\S+", "", text)

    # remove non-words
    text = re.sub(r"[^\w\s]+|\d+|#\S+", "", text)

    # remove additional text (search query)
    text = text.replace(exclude_too, "")

    return text


def analyze(analyzer, text: str, exclude_too: str):
    clean_text = clean(text, exclude_too)
    scores = analyzer.polarity_scores(clean_text)
    return scores["compound"]

In [6]:
node_folder = src.PATH / "data/interim/node_lists/"
node_files = list(node_folder.iterdir())

In [7]:
def roberta_sentiment(text, exclude_too):
    global model, tokenizer, config
    if pd.isnull(text):
        return (np.nan, np.nan)
    text = clean(text, exclude_too)
    out = pipe(text, truncation=True, max_length=512)
    out = out[0]
    return out["label"], out["score"]

In [8]:
out_folder = src.PATH / "data/interim/sentiments/"
out_folder.mkdir(exist_ok=True, parents=True)

In [9]:
for file in tqdm(node_files):
    filename = file.name
    search_query = file.name.replace("_", " ").replace(".csv", "")
    out_file = out_folder / filename

    df = pd.read_csv(file)

    df["sentiment_title_vader"] = df["title"].apply(
        lambda x: analyze(analyzer, x, search_query)
    )

    df["sentiment_title_roberta_label"], df["sentiment_title_roberta_score"] = zip(
        *df["title"].apply(lambda x: roberta_sentiment(x, search_query))
    )

    (
        df["sentiment_description_roberta_label"],
        df["sentiment_description_roberta_score"],
    ) = zip(*df["description"].apply(lambda x: roberta_sentiment(x, search_query)))

    df[
        [
            "video_id",
            "sentiment_title_vader",
            "sentiment_title_roberta_label",
            "sentiment_title_roberta_score",
            "sentiment_description_roberta_label",
            "sentiment_description_roberta_score",
        ]
    ].to_csv(out_file, index=False)

  0%|          | 0/40 [00:00<?, ?it/s]